In [ ]:
# Se cargan tasas de transmisión estimadas antes y después de una intervención ficticia.

import numpy as np
import pandas as pd

observed = pd.read_csv("../data/infection_rates.csv")
observed.tail()

In [ ]:
# ¿Cómo puede retornar gradualmente la tasa reciente a su promedio histórico?

long_run_rate = observed["infection_rate"].mean()
last_rate = observed["infection_rate"].iloc[-1]
mean_reversion = 0.04
forecast_days = np.arange(180)
projected_rates = long_run_rate + (last_rate - long_run_rate) * (1 - mean_reversion) ** forecast_days
rate_forecast = pd.DataFrame({"forecast_day": forecast_days, "infection_rate": projected_rates, "long_run_rate": long_run_rate})
rate_forecast.head()

In [ ]:
# Se implementa un SIR que recibe una tasa de transmisión diferente en cada día.

population = 100_000
initial_infected = 1_000
recovery_rate = 0.08
death_rate = 0.001

def simulate_sir(infection_rates):
    susceptible, infected, recovered, deceased = population - initial_infected, initial_infected, 0.0, 0.0
    records = []
    for day, infection_rate in enumerate(infection_rates):
        records.append({"forecast_day": day, "infection_rate": infection_rate, "susceptible": susceptible, "infected": infected, "recovered": recovered, "deceased": deceased, "required_beds": infected * 0.05})
        new_infections = infection_rate * infected * susceptible / population
        new_recoveries = recovery_rate * infected
        new_deaths = death_rate * infected
        susceptible -= new_infections
        infected += new_infections - new_recoveries - new_deaths
        recovered += new_recoveries
        deceased += new_deaths
    return pd.DataFrame(records)

In [ ]:
# Se compara el pronóstico adaptativo con mantener fija la última tasa observada.

adaptive_forecast = simulate_sir(rate_forecast["infection_rate"])
static_forecast = simulate_sir(np.repeat(last_rate, len(rate_forecast)))
adaptive_forecast["model"] = "tasa_adaptativa"
static_forecast["model"] = "tasa_estatica"
forecasts = pd.concat([static_forecast, adaptive_forecast], ignore_index=True)
forecasts.head()

In [ ]:
# ¿Cómo cambia la evolución esperada cuando la tasa se actualiza en vez de permanecer fija?

import matplotlib.pyplot as plt

figure, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(observed["day"], observed["infection_rate"], marker="o", label="tasa observada")
axes[0].plot(observed["day"].max() + 1 + rate_forecast["forecast_day"], rate_forecast["infection_rate"], label="tasa proyectada")
axes[0].axhline(long_run_rate, color="black", linestyle="--", label="promedio histórico")
axes[0].set(title="Tasa de transmisión adaptativa", xlabel="Día", ylabel="Tasa")
axes[0].grid(alpha=0.3)
axes[0].legend()
for model, forecast in forecasts.groupby("model"):
    axes[1].plot(forecast["forecast_day"], forecast["infected"], label=model.replace("_", " "))
axes[1].set(title="Evolución esperada de casos activos", xlabel="Día desde el corte", ylabel="Casos activos")
axes[1].grid(alpha=0.3)
axes[1].legend()
plt.show()

In [ ]:
# Se conservan la tasa proyectada, los pronósticos y la gráfica para verificar el taller.

from pathlib import Path

submission_dir = Path("../submission")
rate_forecast.to_csv(submission_dir / "infection_rate_forecast.csv", index=False)
forecasts.to_csv(submission_dir / "forecasts.csv", index=False)
figure.savefig(submission_dir / "adaptive_evolution.png", dpi=150, bbox_inches="tight")